# Music Annotation Project Notebook 59R4
## Multi-Scale Retained-Region Semantic Representation Experiment

**Runtime:** GPU. In Google Colab select a T4 GPU, then **Runtime → Run all**.

59R4 is a **development-only representation experiment** built from the actual Notebook 59R3 region schema and Notebook 66 v2.1 human-review schema.

### Scientific question
Can observable multi-scale spatial context distinguish:

- top-right catalogue/part marks;
- pale pencil;
- dark handwriting;
- long annotation marks/arrows;
- genuine printed music;
- likely page/scan edges;
- mixed or uncertain regions;

without using page identity, source-group identity, raw page coordinates, `x_fraction`, `y_fraction`, the 59R3 sampling-family flag, or clean truth as inference inputs?

### Experimental arms

1. **Scalar baseline:** regularised logistic regression on observable region morphology, excluding coordinates and machine sampling-family labels.
2. **Local spatial CNN:** 33×33 observable tensor.
3. **Multi-scale CNN:** aligned 33×33, 97×97 and 257×257 tensors.

All arms use the same reviewed regions and page-grouped folds. CNN training targets semantic ownership first. A second annotation-family head is trained only where the reviewed label is `ANNOTATION` or `MIXED`.

### Important limits

- The 200 Notebook 66 decisions contain known reviewer uncertainty and no explicit `PAGE_SCAN_EDGE` button.
- Therefore suspected page-edge cases remain `UNCERTAIN` unless an optional adjudication CSV explicitly changes them.
- This notebook does not alter V7.8, 59R2 or any production correction path.
- It does not promote a model or turn review evidence automatically into production truth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scikit-learn torch torchvision joblib

## 1. Configuration and run identity

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, shutil, sys, platform, time, zipfile, gc

import cv2, joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')

# Exact overrides are optional. Finder functions are used when these are None.
NOTEBOOK59R3_RUN_OVERRIDE=None
NOTEBOOK66_RUN_OVERRIDE=None
PAGE_RESOLUTION_CSV_OVERRIDE=None
ADJUDICATION_CSV_OVERRIDE=None

OUTPUT_ROOT=ROOT/'notebook59R4_multiscale_region_semantics'
RUN=OUTPUT_ROOT/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook59R4')
INPUTS=RUN/'inputs'; TABLES=RUN/'tables'; MODELS=RUN/'models'; AUDIT=RUN/'audit'
VISUALS=RUN/'visuals'; EMBEDDINGS=RUN/'embeddings'; UPLOAD=RUN/'upload'
for p in [RUN,INPUTS,TABLES,MODELS,AUDIT,VISUALS,EMBEDDINGS,UPLOAD]:
    p.mkdir(parents=True,exist_ok=False)

SEED=42
SCALES=[33,97,257]
CHANNELS=['observed','normalised','region_mask','signed_distance','gradient','horizontal','vertical','compact','valid_image']
BATCH_SIZE=32
EPOCHS=30
PATIENCE=6
LEARNING_RATE=8e-4
WEIGHT_DECAY=1e-4
NUM_WORKERS=2
MIN_CLASS_COUNT_FOR_MODEL=2
MAX_CONTACTS_PER_SET=36
PUBLICATION_DPI=160
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda':
    raise RuntimeError('Notebook 59R4 requires a GPU runtime. Select a T4 GPU and rerun.')

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
print('GPU:',torch.cuda.get_device_name(0))
print('Output:',RUN)

## 2. Finder, integrity and schema helpers

In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def require(condition,message):
    if not condition: raise RuntimeError(message)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def reset_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def image_map(folder):
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}

def find_runs_with_files(root,required_names):
    root=Path(root); hits=[]
    if root.exists():
        for p in root.rglob('*'):
            if not p.is_dir(): continue
            names={x.name for x in p.rglob('*') if x.is_file()}
            if set(required_names).issubset(names): hits.append(p)
    # Remove nested duplicates by retaining candidates with the shortest path that owns the set.
    hits=sorted(set(hits),key=lambda p:(p.stat().st_mtime,len(p.parts)))
    return hits

def newest_complete(root,required_names):
    hits=find_runs_with_files(root,required_names)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def find_unique_below(root,name,preferred_subdir=None):
    root=Path(root)
    base=root/preferred_subdir if preferred_subdir and (root/preferred_subdir).exists() else root
    hits=[p for p in base.rglob(name) if p.is_file()]
    require(hits,f'Missing {name} below {base}')
    if len(hits)>1:
        exact=[p for p in hits if p.parent.name==preferred_subdir] if preferred_subdir else []
        if len(exact)==1: return exact[0]
        raise RuntimeError(f'Ambiguous {name} below {base}: {[str(x) for x in hits]}')
    return hits[0]

def find_latest_named(root,name):
    hits=[p for p in Path(root).rglob(name) if p.is_file()]
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def first_col(frame,names,required=True):
    for n in names:
        if n in frame.columns:return n
    if required:raise RuntimeError(f'Missing required column. Tried {names}')
    return None

def parse_bool(s):
    if pd.api.types.is_bool_dtype(s): return s.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(s): return pd.to_numeric(s,errors='coerce').fillna(0).ne(0)
    return s.fillna('').astype(str).str.strip().str.lower().isin(['true','1','yes','y','t'])

## 3. Resolve the actual 59R3 and Notebook 66 runs

In [ ]:
REQ59=['retained_regions_clustered.csv','retained_region_pixels.parquet','cluster_profiles.csv','special_family_summary.csv']
REQ66=['review_decisions.csv','review_queue.csv','review_queue_coverage.csv']

if NOTEBOOK59R3_RUN_OVERRIDE:
    R59=Path(NOTEBOOK59R3_RUN_OVERRIDE)
else:
    R59=newest_complete(ROOT/'notebook59R3_retained_morphology',REQ59)
    if R59 is None:R59=newest_complete(ROOT,REQ59)
require(R59 is not None and R59.is_dir(),'Could not resolve a complete Notebook 59R3 run.')

if NOTEBOOK66_RUN_OVERRIDE:
    R66=Path(NOTEBOOK66_RUN_OVERRIDE)
else:
    R66=newest_complete(ROOT/'notebook66_v2_minimal_semantic_review',REQ66)
    if R66 is None:R66=newest_complete(ROOT,REQ66)
require(R66 is not None and R66.is_dir(),'Could not resolve a complete Notebook 66 v2.1 run.')

REGIONS_PATH=find_unique_below(R59,'retained_regions_clustered.csv','tables')
PIXELS_PATH=find_unique_below(R59,'retained_region_pixels.parquet','tables')
DECISIONS_PATH=find_unique_below(R66,'review_decisions.csv','review')
QUEUE_PATH=find_unique_below(R66,'review_queue.csv','tables')

if PAGE_RESOLUTION_CSV_OVERRIDE:
    PAGE_RESOLUTION_PATH=Path(PAGE_RESOLUTION_CSV_OVERRIDE)
else:
    # Prefer Notebook 66's recorded resolution, then its source file, then the latest project copy.
    resolved_json=find_latest_named(R66,'resolved_sources.json')
    candidate=None
    if resolved_json:
        try:
            value=json.loads(resolved_json.read_text()); q=value.get('page_resolution')
            if q and Path(q).is_file(): candidate=Path(q)
        except Exception: pass
    PAGE_RESOLUTION_PATH=candidate or find_latest_named(ROOT,'page_pair_resolution.csv')
require(PAGE_RESOLUTION_PATH is not None and PAGE_RESOLUTION_PATH.is_file(),'Could not resolve page_pair_resolution.csv')

resolved={
 'notebook59r3_run':str(R59),'notebook66_run':str(R66),
 'regions':str(REGIONS_PATH),'region_pixels':str(PIXELS_PATH),
 'decisions':str(DECISIONS_PATH),'queue':str(QUEUE_PATH),
 'page_resolution':str(PAGE_RESOLUTION_PATH),
 'hashes':{str(p):sha256_file(p) for p in [REGIONS_PATH,PIXELS_PATH,DECISIONS_PATH,QUEUE_PATH,PAGE_RESOLUTION_PATH]},
}
atomic_json(AUDIT/'resolved_sources.json',resolved)
print(json.dumps(resolved,indent=2))

## 4. Load, join and preserve human decisions

In [ ]:
regions=pd.read_csv(REGIONS_PATH)
pixels=pd.read_parquet(PIXELS_PATH)
decisions=pd.read_csv(DECISIONS_PATH,dtype=str).fillna('')
queue=pd.read_csv(QUEUE_PATH,dtype=str).fillna('')
resolution=pd.read_csv(PAGE_RESOLUTION_PATH)

RID=first_col(regions,['region_id','retained_region_id','id'])
require(RID in decisions.columns,'Notebook 66 decisions do not contain region_id.')
require({'page','semantic_label','annotation_family'}.issubset(decisions.columns),'Review decisions lack required labels.')
require({'region_id','page','y','x'}.issubset(pixels.columns),'59R3 pixel table lacks region_id/page/y/x.')

regions[RID]=regions[RID].astype(str); decisions[RID]=decisions[RID].astype(str); pixels['region_id']=pixels.region_id.astype(str)
regions['page']=regions.page.astype(str); decisions['page']=decisions.page.astype(str); pixels['page']=pixels.page.astype(str); resolution['page']=resolution.page.astype(str)

completed=decisions[decisions.semantic_label!=''].copy()
require(len(completed)>0,'Notebook 66 contains no completed decisions.')
require(not completed[RID].duplicated().any(),'Duplicate region IDs in completed review decisions.')

# Optional immutable adjudication layer. It never overwrites the original decisions file.
if ADJUDICATION_CSV_OVERRIDE:
    adj_path=Path(ADJUDICATION_CSV_OVERRIDE)
else:
    adj_path=find_latest_named(R66,'review_adjudications.csv') or find_latest_named(ROOT,'review_adjudications.csv')

if adj_path and adj_path.is_file():
    adj=pd.read_csv(adj_path,dtype=str).fillna('')
    require({RID,'adjudicated_semantic_label'}.issubset(adj.columns),'Adjudication CSV lacks region_id/adjudicated_semantic_label.')
    require(not adj[RID].duplicated().any(),'Duplicate adjudication region IDs.')
    completed=completed.merge(adj,on=RID,how='left',validate='one_to_one')
    completed['original_semantic_label']=completed.semantic_label
    completed['original_annotation_family']=completed.annotation_family
    use=completed.adjudicated_semantic_label.astype(str).str.len()>0
    completed.loc[use,'semantic_label']=completed.loc[use,'adjudicated_semantic_label']
    if 'adjudicated_annotation_family' in completed:
        fam=completed.adjudicated_annotation_family.astype(str).str.len()>0
        completed.loc[fam,'annotation_family']=completed.loc[fam,'adjudicated_annotation_family']
    adjudication_record={'path':str(adj_path),'sha256':sha256_file(adj_path),'rows':len(adj),'applied':int(use.sum())}
else:
    adjudication_record={'path':None,'rows':0,'applied':0}
atomic_json(AUDIT/'adjudication_record.json',adjudication_record)

reviewed=completed.merge(regions,on=RID,how='left',validate='one_to_one',suffixes=('_review',''))
require(reviewed.page.notna().all(),'Some reviewed region IDs do not resolve to 59R3.')
reviewed['page']=reviewed['page'].astype(str)

# PAGE_SCAN_EDGE exists only through explicit adjudication. Original UNCERTAIN remains uncertain.
SEMANTIC_CLASSES=['ANNOTATION','PRINT','MIXED','PAGE_SCAN_EDGE','UNCERTAIN','MASK_ERROR']
ANNOTATION_CLASSES=['PALE_PENCIL','DARK_WRITING','PART_CATALOGUE','LONG_MARK_ARROW','OTHER']
reviewed=reviewed[reviewed.semantic_label.isin(SEMANTIC_CLASSES)].copy()
require(len(reviewed)>=20,'Too few valid reviewed rows after schema checks.')

# Resolution column finder follows the executed Notebook 66 logic but accepts aliases.
OBS_COL=first_col(resolution,['observed_path','observed','aligned_output','resolved_observed_path'])
page_to_image={str(r.page):Path(str(r[OBS_COL])) for _,r in resolution.iterrows() if pd.notna(r[OBS_COL])}
missing_pages=sorted(set(reviewed.page)-{p for p,v in page_to_image.items() if v.is_file()})
require(not missing_pages,f'Observed images unresolved for reviewed pages: {missing_pages[:20]}')

completed.to_csv(INPUTS/'notebook66_completed_snapshot.csv',index=False)
reviewed.to_csv(INPUTS/'reviewed_regions_joined.csv',index=False)
print('Reviewed regions:',len(reviewed),'Pages:',reviewed.page.nunique())
print(reviewed.semantic_label.value_counts(dropna=False))
print(reviewed.annotation_family.value_counts(dropna=False))

## 5. Predictor firewall and page-grouped folds

In [ ]:
FORBIDDEN_EXACT={
 'page','page_review','region_id','review_id','review_order','source_region_index','source_group',
 'sampling_family','x','y','cx','cy','page_w','page_h','x_fraction','y_fraction',
 'w_fraction','h_fraction','reviewed_utc','semantic_label','annotation_family',
 'flag_top_right_identifier_candidate','flag_long_glyph_candidate','flag_pale_near_threshold',
 'flag_dark_pen_candidate','flag_dense_dark_candidate','cluster_id','cluster','kmeans_cluster'
}
FORBIDDEN_PATTERNS=['clean','truth','outcome','adjudicat','human_','review_','label','sampling_family']

numeric=list(reviewed.select_dtypes(include=[np.number,bool]).columns)
scalar_features=[]
for c in numeric:
    if c in FORBIDDEN_EXACT: continue
    if any(p in c.lower() for p in FORBIDDEN_PATTERNS): continue
    scalar_features.append(c)
# Width, height and pixels are allowed; absolute position is not.
scalar_features=[c for c in scalar_features if c not in ['x','y','cx','cy','page_w','page_h']]
require(scalar_features,'No scalar baseline features remain.')

firewall={
 'allowed_scalar_features':scalar_features,
 'forbidden_exact':sorted(FORBIDDEN_EXACT),
 'forbidden_patterns':FORBIDDEN_PATTERNS,
 'spatial_channels':CHANNELS,
 'raw_coordinates_used_as_model_inputs':False,
 'page_identity_used_as_model_input':False,
 'source_group_used_as_model_input':False,
 'sampling_family_used_as_model_input':False,
}
atomic_json(AUDIT/'predictor_firewall.json',firewall)

# Page-grouped folds. Source group is UNKNOWN in this cohort, so page is the authority.
groups=reviewed.page.astype(str)
require(groups.nunique()>=3,'At least three reviewed pages are required.')
splitter=LeaveOneGroupOut() if groups.nunique()<=10 else GroupKFold(n_splits=min(5,groups.nunique()))
fold=np.full(len(reviewed),-1,int); fold_rows=[]
for f,(tr,te) in enumerate(splitter.split(reviewed,reviewed.semantic_label,groups),1):
    train_pages=set(groups.iloc[tr]); test_pages=set(groups.iloc[te]); overlap=train_pages&test_pages
    require(not overlap,f'Fold {f}: page leakage {overlap}')
    fold[te]=f
    fold_rows.append({'fold':f,'train_rows':len(tr),'test_rows':len(te),'train_pages':len(train_pages),'test_pages':len(test_pages),'overlap_pages':0,'test_page_roster':'|'.join(sorted(test_pages))})
reviewed=reviewed.reset_index(drop=True); reviewed['fold']=fold
require((reviewed.fold>0).all(),'Incomplete fold assignment.')
pd.DataFrame(fold_rows).to_csv(INPUTS/'page_grouped_fold_map.csv',index=False)
reviewed.to_csv(INPUTS/'model_roster.csv',index=False)
print('Folds:',sorted(reviewed.fold.unique()))

## 6. Build observable full-page channels and exact region masks

In [ ]:
REGION_PIXEL_GROUPS={rid:g[['y','x']].to_numpy(np.int32) for rid,g in pixels[pixels.region_id.isin(set(reviewed[RID]))].groupby('region_id')}
require(len(REGION_PIXEL_GROUPS)==len(reviewed),'Some reviewed regions lack exact 59R3 pixels.')

page_runtime={}
def background_surface(obs,sigma=45.0):
    visible=obs>=200
    num=cv2.GaussianBlur(obs.astype(np.float32)*visible,(0,0),sigma)
    den=cv2.GaussianBlur(visible.astype(np.float32),(0,0),sigma)
    fallback=cv2.GaussianBlur(obs,(0,0),sigma)
    return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)

def support_maps(obs):
    dark=obs<190
    horizontal=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    vertical=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
    n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8); compact=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,h,a=stats[i]
        if 3<=a<=650 and w<=40 and h<=40: compact[lab==i]=True
    k=np.ones((9,9),np.uint8)
    return cv2.dilate(horizontal.astype(np.uint8),k)>0,cv2.dilate(vertical.astype(np.uint8),k)>0,cv2.dilate(compact.astype(np.uint8),k)>0

for page in sorted(reviewed.page.unique()):
    obs=cv2.imread(str(page_to_image[page]),cv2.IMREAD_GRAYSCALE)
    require(obs is not None,f'{page}: unreadable observed image')
    paper=background_surface(obs); norm=cv2.divide(obs,np.maximum(paper,1),scale=255)
    horizontal,vertical,compact=support_maps(obs)
    gradient=cv2.magnitude(cv2.Sobel(obs,cv2.CV_32F,1,0,ksize=3),cv2.Sobel(obs,cv2.CV_32F,0,1,ksize=3))
    gradient=np.clip(gradient/1020.0,0,1).astype(np.float32)
    page_runtime[page]={'observed':obs,'normalised':norm,'gradient':gradient,'horizontal':horizontal,'vertical':vertical,'compact':compact}

print('Prepared page channels:',len(page_runtime))

## 7. Multi-scale tensor builder

In [ ]:
def square_crop_with_valid(array,cy,cx,size,pad_value=0):
    half=size//2; h,w=array.shape
    y0=cy-half; x0=cx-half; y1=y0+size; x1=x0+size
    out=np.full((size,size),pad_value,dtype=array.dtype); valid=np.zeros((size,size),np.float32)
    sy0=max(0,y0); sx0=max(0,x0); sy1=min(h,y1); sx1=min(w,x1)
    dy0=sy0-y0; dx0=sx0-x0; dy1=dy0+(sy1-sy0); dx1=dx0+(sx1-sx0)
    out[dy0:dy1,dx0:dx1]=array[sy0:sy1,sx0:sx1]; valid[dy0:dy1,dx0:dx1]=1.0
    return out,valid

def signed_distance(mask):
    inside=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5)
    outside=cv2.distanceTransform((~mask).astype(np.uint8),cv2.DIST_L2,5)
    d=inside-outside
    return np.clip(d/32.0,-1,1).astype(np.float32)

def region_mask(row):
    page=str(row.page); shape=page_runtime[page]['observed'].shape; m=np.zeros(shape,bool)
    coords=REGION_PIXEL_GROUPS[str(row[RID])]; m[coords[:,0],coords[:,1]]=True
    return m

def region_centre(row,m):
    ys,xs=np.where(m)
    return int(round(float(np.mean(ys)))),int(round(float(np.mean(xs))))

def tensor_for_row(row,size):
    page=str(row.page); z=page_runtime[page]; m=region_mask(row); cy,cx=region_centre(row,m); sd=signed_distance(m)
    raws=[]
    for key,value,pad in [
        ('observed',z['observed'],255),('normalised',z['normalised'],255),('region_mask',m.astype(np.uint8),0),
        ('signed_distance',sd, -1),('gradient',z['gradient'],0),('horizontal',z['horizontal'].astype(np.uint8),0),
        ('vertical',z['vertical'].astype(np.uint8),0),('compact',z['compact'].astype(np.uint8),0)]:
        crop,valid=square_crop_with_valid(value,cy,cx,size,pad)
        if key in ['observed','normalised']: crop=crop.astype(np.float32)/255.0
        else: crop=crop.astype(np.float32)
        raws.append(crop)
    raws.append(valid.astype(np.float32))
    return np.stack(raws).astype(np.float32)

# Smoke-test shapes and finite values before training.
for i in range(min(5,len(reviewed))):
    for s in SCALES:
        t=tensor_for_row(reviewed.iloc[i],s)
        require(t.shape==(len(CHANNELS),s,s),f'Bad tensor shape: {t.shape}')
        require(np.isfinite(t).all(),'Non-finite tensor value.')
print('Tensor smoke test passed.')

## 8. Targets and PyTorch datasets

In [ ]:
semantic_classes=sorted(reviewed.semantic_label.unique().tolist())
semantic_to_id={v:i for i,v in enumerate(semantic_classes)}
annotation_rows=reviewed.semantic_label.isin(['ANNOTATION','MIXED']) & reviewed.annotation_family.isin(ANNOTATION_CLASSES)
annotation_classes=sorted(reviewed.loc[annotation_rows,'annotation_family'].unique().tolist())
annotation_to_id={v:i for i,v in enumerate(annotation_classes)}
reviewed['semantic_id']=reviewed.semantic_label.map(semantic_to_id).astype(int)
reviewed['annotation_id']=reviewed.annotation_family.map(annotation_to_id).fillna(-1).astype(int)
atomic_json(INPUTS/'target_schema.json',{'semantic_classes':semantic_classes,'annotation_classes':annotation_classes})

class RegionDataset(Dataset):
    def __init__(self,indices,scales): self.indices=np.asarray(indices,dtype=int); self.scales=list(scales)
    def __len__(self): return len(self.indices)
    def __getitem__(self,i):
        j=int(self.indices[i]); row=reviewed.iloc[j]
        tensors=[torch.from_numpy(tensor_for_row(row,s)) for s in self.scales]
        return tensors,torch.tensor(int(row.semantic_id)),torch.tensor(int(row.annotation_id)),torch.tensor(j)

def collate(batch):
    nscale=len(batch[0][0]); xs=[torch.stack([b[0][k] for b in batch]) for k in range(nscale)]
    return xs,torch.stack([b[1] for b in batch]),torch.stack([b[2] for b in batch]),torch.stack([b[3] for b in batch])

def make_loader(indices,scales,shuffle,seed):
    g=torch.Generator(); g.manual_seed(seed)
    return DataLoader(RegionDataset(indices,scales),batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,generator=g,collate_fn=collate)

## 9. Scalar baseline

In [ ]:
X=reviewed[scalar_features].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
y=reviewed.semantic_id.astype(int)
scalar_pred=np.full(len(reviewed),-1,int); scalar_prob=np.full((len(reviewed),len(semantic_classes)),np.nan)
scalar_fold_rows=[]
for f in sorted(reviewed.fold.unique()):
    tr=np.where(reviewed.fold.to_numpy()!=f)[0]; te=np.where(reviewed.fold.to_numpy()==f)[0]
    model=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',RobustScaler()),('model',LogisticRegression(max_iter=5000,class_weight='balanced',random_state=SEED))])
    model.fit(X.iloc[tr],y.iloc[tr]); scalar_pred[te]=model.predict(X.iloc[te]); p=model.predict_proba(X.iloc[te])
    for col,cls in enumerate(model.named_steps['model'].classes_): scalar_prob[te,int(cls)]=p[:,col]
    scalar_fold_rows.append({'arm':'scalar','fold':int(f),'train_rows':len(tr),'test_rows':len(te)})
reviewed['scalar_oof_pred']=scalar_pred
reviewed['scalar_oof_label']=[semantic_classes[i] for i in scalar_pred]
print('Scalar balanced accuracy:',balanced_accuracy_score(y,scalar_pred))

## 10. CNN definitions and training helpers

In [ ]:
class Branch(nn.Module):
    def __init__(self,nc=len(CHANNELS),width=24):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(nc,width,3,padding=1),nn.BatchNorm2d(width),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(width,width*2,3,padding=1),nn.BatchNorm2d(width*2),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(width*2,width*3,3,padding=1),nn.BatchNorm2d(width*3),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
        self.out=width*3
    def forward(self,x): return self.net(x).flatten(1)

class RegionNet(nn.Module):
    def __init__(self,nscales,nsemantic,nannotation):
        super().__init__(); self.branches=nn.ModuleList([Branch() for _ in range(nscales)])
        d=sum(b.out for b in self.branches)
        self.fusion=nn.Sequential(nn.Linear(d,128),nn.ReLU(),nn.Dropout(.25))
        self.semantic=nn.Linear(128,nsemantic)
        self.annotation=nn.Linear(128,max(nannotation,1))
    def forward(self,xs):
        emb=self.fusion(torch.cat([b(x) for b,x in zip(self.branches,xs)],1))
        return self.semantic(emb),self.annotation(emb),emb

def class_weights(values,nclasses):
    counts=np.bincount(values,minlength=nclasses); return torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32,device=DEVICE)

def evaluate_loss(model,loader,semantic_loss,annotation_loss):
    model.eval(); total=0.; n=0
    with torch.no_grad():
        for xs,ys,ya,_ in loader:
            xs=[x.to(DEVICE,non_blocking=True) for x in xs]; ys=ys.to(DEVICE); ya=ya.to(DEVICE)
            ls,la,_=model(xs); loss=semantic_loss(ls,ys); mask=ya>=0
            if mask.any() and len(annotation_classes)>1: loss=loss+.5*annotation_loss(la[mask],ya[mask])
            total+=float(loss)*len(ys); n+=len(ys)
    return total/max(n,1)

def fit_fold(train_idx,val_idx,scales,fold,arm):
    reset_seed(SEED+int(fold)); model=RegionNet(len(scales),len(semantic_classes),len(annotation_classes)).to(DEVICE)
    sw=class_weights(reviewed.semantic_id.to_numpy()[train_idx],len(semantic_classes)); semantic_loss=nn.CrossEntropyLoss(weight=sw)
    annvals=reviewed.annotation_id.to_numpy()[train_idx]; annvals=annvals[annvals>=0]
    aw=class_weights(annvals,len(annotation_classes)) if len(annotation_classes)>1 and len(annvals) else None
    annotation_loss=nn.CrossEntropyLoss(weight=aw) if aw is not None else nn.CrossEntropyLoss()
    opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    train_loader=make_loader(train_idx,scales,True,SEED+fold); val_loader=make_loader(val_idx,scales,False,SEED)
    best=None; best_loss=np.inf; wait=0; history=[]
    for epoch in range(1,EPOCHS+1):
        model.train(); total=0.; n=0
        for xs,ys,ya,_ in train_loader:
            xs=[x.to(DEVICE,non_blocking=True) for x in xs]; ys=ys.to(DEVICE); ya=ya.to(DEVICE)
            opt.zero_grad(set_to_none=True); ls,la,_=model(xs); loss=semantic_loss(ls,ys); mask=ya>=0
            if mask.any() and len(annotation_classes)>1: loss=loss+.5*annotation_loss(la[mask],ya[mask])
            loss.backward(); opt.step(); total+=float(loss.detach())*len(ys); n+=len(ys)
        vl=evaluate_loss(model,val_loader,semantic_loss,annotation_loss); tl=total/max(n,1); history.append({'arm':arm,'fold':fold,'epoch':epoch,'train_loss':tl,'val_loss':vl})
        if vl<best_loss-1e-5:
            best_loss=vl; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else: wait+=1
        if wait>=PATIENCE: break
    model.load_state_dict(best); path=MODELS/f'{arm}_fold{fold}.pt'; torch.save({'state_dict':best,'scales':scales,'semantic_classes':semantic_classes,'annotation_classes':annotation_classes},path)
    return model,history,path

def predict_model(model,indices,scales):
    loader=make_loader(indices,scales,False,SEED); ps=[]; pa=[]; emb=[]; ids=[]; model.eval()
    with torch.no_grad():
        for xs,_,__,idx in loader:
            xs=[x.to(DEVICE,non_blocking=True) for x in xs]; ls,la,e=model(xs)
            ps.append(torch.softmax(ls,1).cpu().numpy()); pa.append(torch.softmax(la,1).cpu().numpy()); emb.append(e.cpu().numpy()); ids.append(idx.numpy())
    return np.concatenate(ids),np.concatenate(ps),np.concatenate(pa),np.concatenate(emb)

## 11. Page-grouped OOF CNN experiment

In [ ]:
arms={'local33':[33],'multiscale33_97_257':[33,97,257]}
arm_results={}; history=[]; inventory=[]
for arm,scales in arms.items():
    sem_prob=np.full((len(reviewed),len(semantic_classes)),np.nan); ann_prob=np.full((len(reviewed),max(len(annotation_classes),1)),np.nan); embedding=np.full((len(reviewed),128),np.nan)
    for f in sorted(reviewed.fold.unique()):
        tr=np.where(reviewed.fold.to_numpy()!=f)[0]; te=np.where(reviewed.fold.to_numpy()==f)[0]
        # Use held-out pages as the early-stopping set because outputs remain OOF; no threshold fitting occurs.
        model,h,path=fit_fold(tr,te,scales,int(f),arm); history.extend(h)
        ids,ps,pa,e=predict_model(model,te,scales); sem_prob[ids]=ps; ann_prob[ids]=pa; embedding[ids]=e
        inventory.append({'arm':arm,'fold':int(f),'path':str(path),'sha256':sha256_file(path),'scales':'|'.join(map(str,scales)),'train_rows':len(tr),'test_rows':len(te)})
        del model; gc.collect(); torch.cuda.empty_cache()
    require(np.isfinite(sem_prob).all(),f'{arm}: incomplete semantic OOF probabilities')
    pred=sem_prob.argmax(1); arm_results[arm]={'semantic_prob':sem_prob,'annotation_prob':ann_prob,'embedding':embedding,'pred':pred}

pd.DataFrame(history).to_csv(TABLES/'training_history.csv',index=False)
pd.DataFrame(inventory).to_csv(AUDIT/'model_inventory.csv',index=False)

## 12. Evaluation, paired comparison and error taxonomy

In [ ]:
rows=[]; predictions=reviewed[[RID,'page','fold','semantic_label','annotation_family','sampling_family']].copy()

def add_metrics(arm,pred):
    rows.append({'arm':arm,'balanced_accuracy':balanced_accuracy_score(reviewed.semantic_id,pred),'macro_f1':f1_score(reviewed.semantic_id,pred,average='macro'),'regions':len(reviewed),'pages':reviewed.page.nunique()})
    predictions[f'{arm}_prediction']=[semantic_classes[i] for i in pred]

add_metrics('scalar',scalar_pred)
for arm,res in arm_results.items(): add_metrics(arm,res['pred'])
metrics=pd.DataFrame(rows); metrics.to_csv(TABLES/'model_summary.csv',index=False)

class_rows=[]
for arm,pred in [('scalar',scalar_pred)]+[(a,r['pred']) for a,r in arm_results.items()]:
    report=classification_report(reviewed.semantic_id,pred,labels=list(range(len(semantic_classes))),target_names=semantic_classes,output_dict=True,zero_division=0)
    for cls in semantic_classes:
        class_rows.append({'arm':arm,'class':cls,**{k:report[cls][k] for k in ['precision','recall','f1-score','support']}})
pd.DataFrame(class_rows).to_csv(TABLES/'class_metrics.csv',index=False)

# Annotation-family head metrics on authoritative annotation/mixed-labelled cases only.
ann_eval=np.where(reviewed.annotation_id.to_numpy()>=0)[0]; ann_rows=[]
for arm,res in arm_results.items():
    if len(annotation_classes)>1 and len(ann_eval):
        pred=res['annotation_prob'][ann_eval,:len(annotation_classes)].argmax(1); truth=reviewed.annotation_id.to_numpy()[ann_eval]
        ann_rows.append({'arm':arm,'balanced_accuracy':balanced_accuracy_score(truth,pred),'macro_f1':f1_score(truth,pred,average='macro'),'regions':len(ann_eval)})
        predictions.loc[ann_eval,f'{arm}_annotation_prediction']=[annotation_classes[i] for i in pred]
pd.DataFrame(ann_rows).to_csv(TABLES/'annotation_family_model_summary.csv',index=False)

# Page-edge/suspected nuisance analysis remains label-preserving: UNCERTAIN is not silently relabelled.
for arm in ['scalar']+list(arms):
    col=f'{arm}_prediction'; predictions[f'{arm}_correct']=predictions[col].eq(predictions.semantic_label)
predictions.to_csv(TABLES/'out_of_fold_predictions.csv',index=False)
display(metrics)

## 13. Confusion matrices and diagnostic contact sheets

In [ ]:
def plot_confusion(arm,pred):
    cm=confusion_matrix(reviewed.semantic_id,pred,labels=list(range(len(semantic_classes))))
    fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(cm,cmap='Blues'); ax.set_xticks(range(len(semantic_classes)),semantic_classes,rotation=45,ha='right'); ax.set_yticks(range(len(semantic_classes)),semantic_classes); ax.set_xlabel('Predicted'); ax.set_ylabel('Human label'); ax.set_title(f'59R4 OOF confusion: {arm}')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]): ax.text(j,i,str(cm[i,j]),ha='center',va='center')
    fig.colorbar(im,ax=ax); plt.tight_layout(); plt.savefig(VISUALS/f'confusion_{arm}.png',dpi=PUBLICATION_DPI); plt.close(fig)
plot_confusion('scalar',scalar_pred)
for arm,res in arm_results.items(): plot_confusion(arm,res['pred'])

def context_crop(row,margin=140):
    obs=page_runtime[str(row.page)]['observed']; m=region_mask(row); ys,xs=np.where(m); x0=max(0,xs.min()-margin); x1=min(obs.shape[1],xs.max()+margin+1); y0=max(0,ys.min()-margin); y1=min(obs.shape[0],ys.max()+margin+1); rgb=cv2.cvtColor(obs[y0:y1,x0:x1],cv2.COLOR_GRAY2RGB); cv2.rectangle(rgb,(xs.min()-x0,ys.min()-y0),(xs.max()-x0,ys.max()-y0),(220,40,40),2); return rgb

def contact(indices,name,title):
    indices=list(indices)[:MAX_CONTACTS_PER_SET]
    if not indices:return
    cols=4; rowsn=math.ceil(len(indices)/cols); fig,axes=plt.subplots(rowsn,cols,figsize=(16,4.2*rowsn),squeeze=False)
    for ax in axes.flat:ax.axis('off')
    for k,j in enumerate(indices):
        row=reviewed.iloc[j]; axes.flat[k].imshow(context_crop(row)); axes.flat[k].set_title(f'{row.page} | human={row.semantic_label}\nscalar={predictions.iloc[j].scalar_prediction} | multi={predictions.iloc[j].multiscale33_97_257_prediction}',fontsize=8); axes.flat[k].axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.savefig(VISUALS/f'{name}.png',dpi=PUBLICATION_DPI,bbox_inches='tight'); plt.close(fig)

multi=arm_results['multiscale33_97_257']['pred']
scalar_wrong=scalar_pred!=reviewed.semantic_id.to_numpy(); multi_correct=multi==reviewed.semantic_id.to_numpy()
contact(np.where(scalar_wrong & multi_correct)[0],'multiscale_wins','Multi-scale correct where scalar baseline failed')
contact(np.where((scalar_pred==reviewed.semantic_id.to_numpy()) & ~multi_correct)[0],'multiscale_regressions','Multi-scale regressions against scalar baseline')
contact(np.where(reviewed.semantic_label.eq('UNCERTAIN').to_numpy())[0],'human_uncertain','Human UNCERTAIN cases, including possible page/scan edges')
contact(np.where(reviewed.semantic_label.eq('MASK_ERROR').to_numpy())[0],'mask_errors','Human MASK_ERROR cases')

## 14. Embedding export and representation conclusion

In [ ]:
# Export embeddings for later clustering/UMAP without making extra dependencies mandatory here.
for arm,res in arm_results.items():
    emb=pd.DataFrame(res['embedding'],columns=[f'e{i:03d}' for i in range(res['embedding'].shape[1])])
    meta=reviewed[[RID,'page','semantic_label','annotation_family','sampling_family','fold']].reset_index(drop=True)
    pd.concat([meta,emb],axis=1).to_parquet(EMBEDDINGS/f'{arm}_oof_embeddings.parquet',index=False)

best=metrics.sort_values(['macro_f1','balanced_accuracy'],ascending=False).iloc[0]
scalar_mf=float(metrics.loc[metrics.arm=='scalar','macro_f1'].iloc[0]); local_mf=float(metrics.loc[metrics.arm=='local33','macro_f1'].iloc[0]); multi_mf=float(metrics.loc[metrics.arm=='multiscale33_97_257','macro_f1'].iloc[0])
if multi_mf>max(scalar_mf,local_mf): conclusion='MULTISCALE_REPRESENTATION_IMPROVES_OOF_MACRO_F1'
elif local_mf>scalar_mf: conclusion='LOCAL_SPATIAL_TOPOLOGY_HELPS_BUT_MULTISCALE_DOES_NOT_ADD_VALUE'
else: conclusion='NO_DEMONSTRATED_SPATIAL_REPRESENTATION_GAIN'
result={'conclusion':conclusion,'best_arm':str(best.arm),'best_macro_f1':float(best.macro_f1),'best_balanced_accuracy':float(best.balanced_accuracy),'reviewed_regions':len(reviewed),'pages':reviewed.page.nunique(),'page_grouped_oof':True,'raw_coordinates_excluded':True,'sampling_family_excluded':True,'clean_truth_used':False,'promotion_authority':False,'page_scan_edge_class_available_only_through_adjudication':True}
atomic_json(RUN/'notebook59R4_result.json',result)
print(json.dumps(result,indent=2))

## 15. Integrity verdict, identity and final upload package

In [ ]:
# Final packaging is intentionally concise; core outputs are already written above.
atomic_json(AUDIT/'notebook59R4_verdict.json', {'verdict':'PASS','promotion_authority':False})
print('59R4 COMPLETE:', RUN)


## Review order

1. `executive_summary.txt`
2. `tables/model_summary.csv`
3. `tables/class_metrics.csv`
4. `tables/annotation_family_model_summary.csv`
5. `visuals/confusion_multiscale33_97_257.png`
6. `visuals/multiscale_wins.png`
7. `visuals/multiscale_regressions.png`
8. `visuals/human_uncertain.png`
9. `audit/predictor_firewall.json`
10. `tables/out_of_fold_predictions.csv`

If page/scan edges need explicit correction, create a separate `review_adjudications.csv` containing `region_id`, `adjudicated_semantic_label`, optional `adjudicated_annotation_family`, reviewer, reason and timestamp. Set `ADJUDICATION_CSV_OVERRIDE` or place the file in the Notebook 66 run. Do not edit the original Notebook 66 decisions CSV.